# 🔐 SigAuth Complete Backend v8 — Production-Ready

## What's Fixed vs v7
1. **Pretrained ResNet-18** backbone (grayscale→3ch) — massive feature quality improvement
2. **Triplet Loss** instead of Contrastive — creates larger genuine/forged embedding gap
3. **No SVC mixing into tamper dataset** — prevents domain confusion that caused false "tampered" on real signatures
4. **Simple hard-rule fusion** — no arbitrary softmax coefficients, just calibrated thresholds
5. **All missing API endpoints** — `/auth/profile`, `/admin/logs`, `DELETE /admin/signatures/<id>`
6. **Stronger data augmentation** for both models

## Before Running
1. Set runtime to **GPU** (Runtime → Change runtime type → T4 GPU)
2. Upload these 3 ZIP files to `/content/` via the Files panel:
   - `task1.zip`
   - `task2.zip`  
   - `synthetic_tamper.zip`
3. Run all cells in order
4. Copy the ngrok URL into your `.env.local` as `VITE_API_URL=<url>`

## ⚠️ Colab Limitation
The website only works while Colab is running because the Flask server lives here.
To make it permanent, deploy to Render/Railway/any VPS.


In [ ]:
# ╔══════════════════════════════════════╗
# ║  CELL 1: Install & Setup             ║
# ╚══════════════════════════════════════╝

!pip -q install flask flask-cors pyngrok pillow scikit-learn matplotlib seaborn tqdm pyjwt

import os, io, re, time, json, math, base64, random, zipfile, sqlite3, hashlib, warnings
from pathlib import Path
from typing import List, Tuple, Dict, Optional
from collections import defaultdict

import numpy as np
from PIL import Image, ImageFilter, ImageOps, ImageEnhance
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split, Subset
import torchvision.transforms as T
import torchvision.models as models

from sklearn.metrics import (
    classification_report, confusion_matrix, roc_curve, auc,
    precision_recall_curve, f1_score, accuracy_score
)

warnings.filterwarnings('ignore')

# ── Reproducibility ──
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'🖥️ Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'   GPU: {torch.cuda.get_device_name(0)}')

# ── Configuration (edit these to fine-tune) ──
CFG = {
    'img_size': 224,
    'batch_size': 32,
    'workers': 2,
    # Siamese (Triplet Loss)
    'siamese_epochs': 40,
    'siamese_lr': 3e-4,
    'triplet_margin': 0.5,
    'embedding_dim': 256,
    # Tamper CNN
    'tamper_epochs': 50,
    'tamper_lr': 1e-4,
    # Shared
    'weight_decay': 1e-4,
    'early_stop_patience': 10,
    'triplets_per_user': 150,   # more triplets = better separation
}

os.makedirs('datasets/svc2004/task1', exist_ok=True)
os.makedirs('datasets/svc2004/task2', exist_ok=True)
os.makedirs('datasets/tamper', exist_ok=True)
os.makedirs('saved_models', exist_ok=True)
os.makedirs('results', exist_ok=True)

print('✅ Setup complete')
print(f'📋 Config: {json.dumps(CFG, indent=2)}')


In [ ]:
# ╔══════════════════════════════════════╗
# ║  CELL 2: Extract Datasets            ║
# ╚══════════════════════════════════════╝

def extract_zip(candidates, target):
    """Try extracting from multiple candidate paths."""
    for p in candidates:
        if os.path.exists(p):
            with zipfile.ZipFile(p, 'r') as zf:
                zf.extractall(target)
            print(f'  ✅ Extracted {p} → {target}')
            return True
    print(f'  ⚠️ Not found: {candidates}')
    return False

print('📦 Extracting datasets...')
extract_zip(['task1.zip', '/content/task1.zip', 'Task1.zip', '/content/Task1.zip'],
            'datasets/svc2004/task1')
extract_zip(['task2.zip', '/content/task2.zip', 'Task2.zip', '/content/Task2.zip'],
            'datasets/svc2004/task2')
extract_zip(['synthetic_tamper.zip', '/content/synthetic_tamper.zip',
             'Synthetic_Tamper.zip', '/content/Synthetic_Tamper.zip'],
            'datasets/tamper')

# ── Find nested directories ──
def find_dir(base, name):
    """Recursively find a directory by name."""
    for root, dirs, _ in os.walk(base):
        if name in dirs:
            return os.path.join(root, name)
    return os.path.join(base, name)

CLEAN_DIR = find_dir('datasets/tamper', 'clean')
TAMPER_DIR = find_dir('datasets/tamper', 'tampered')

# ── Collect all image paths ──
def list_images(folder):
    exts = {'.png', '.jpg', '.jpeg', '.bmp', '.tiff'}
    out = []
    if os.path.isdir(folder):
        for root, _, files in os.walk(folder):
            for f in files:
                if Path(f).suffix.lower() in exts and not f.startswith('.'):
                    out.append(os.path.join(root, f))
    return sorted(out)

task1_imgs = list_images('datasets/svc2004/task1')
task2_imgs = list_images('datasets/svc2004/task2')
clean_imgs = list_images(CLEAN_DIR)
tampered_imgs = list_images(TAMPER_DIR)

print(f'\n📊 Dataset Summary:')
print(f'  SVC Task1: {len(task1_imgs)} images')
print(f'  SVC Task2: {len(task2_imgs)} images')
print(f'  Clean (tamper): {len(clean_imgs)} from {CLEAN_DIR}')
print(f'  Tampered: {len(tampered_imgs)} from {TAMPER_DIR}')

assert len(task1_imgs) > 0, 'ERROR: task1.zip not found or empty!'
assert len(task2_imgs) > 0, 'ERROR: task2.zip not found or empty!'
assert len(clean_imgs) > 0, 'ERROR: synthetic_tamper.zip clean/ not found!'
assert len(tampered_imgs) > 0, 'ERROR: synthetic_tamper.zip tampered/ not found!'
print('\n✅ All datasets loaded successfully')


In [ ]:
# ╔══════════════════════════════════════╗
# ║  CELL 3: Parse SVC + Build Datasets  ║
# ╚══════════════════════════════════════╝

# ── Parse SVC2004: S1-S20 = genuine, S21-S40 = forged ──
IMG_RE = re.compile(r'U(\d+)S(\d+)', re.IGNORECASE)

def parse_svc(images):
    users = defaultdict(lambda: {'genuine': [], 'forged': []})
    for p in images:
        m = IMG_RE.search(Path(p).stem)
        if not m:
            continue
        uid = int(m.group(1))
        sid = int(m.group(2))
        if 1 <= sid <= 20:
            users[uid]['genuine'].append(p)
        elif 21 <= sid <= 40:
            users[uid]['forged'].append(p)
    # Keep users with enough samples
    valid = {u: d for u, d in users.items()
             if len(d['genuine']) >= 5 and len(d['forged']) >= 5}
    return valid

svc_users = parse_svc(task1_imgs + task2_imgs)
print(f'📝 Valid SVC users: {len(svc_users)}')

total_genuine = sum(len(d['genuine']) for d in svc_users.values())
total_forged = sum(len(d['forged']) for d in svc_users.values())
print(f'   Genuine samples: {total_genuine}')
print(f'   Forged samples: {total_forged}')

# ── Split users into train/val (80/20 by user, not by sample) ──
all_uids = sorted(svc_users.keys())
random.shuffle(all_uids)
split_idx = int(0.8 * len(all_uids))
train_uids = set(all_uids[:split_idx])
val_uids = set(all_uids[split_idx:])
print(f'   Train users: {len(train_uids)}, Val users: {len(val_uids)}')

# ── Transforms ──
# We use 3-channel input to leverage pretrained ResNet features
train_tf = T.Compose([
    T.Resize((CFG['img_size'], CFG['img_size'])),
    T.RandomAffine(degrees=5, translate=(0.05, 0.05), scale=(0.92, 1.08), shear=3),
    T.RandomPerspective(distortion_scale=0.1, p=0.3),
    T.ColorJitter(brightness=0.15, contrast=0.15),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_tf = T.Compose([
    T.Resize((CFG['img_size'], CFG['img_size'])),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def load_img_3ch(path):
    """Load image and convert to 3-channel RGB for pretrained model."""
    img = Image.open(path).convert('L')  # grayscale first
    return Image.merge('RGB', (img, img, img))  # replicate to 3 channels

# ── Triplet Dataset for Siamese ──
class TripletDataset(Dataset):
    """Generates (anchor, positive, negative) triplets for each user.
    Anchor & Positive = genuine from same user.
    Negative = forged from same user (hard negatives for skilled forgery).
    """
    def __init__(self, user_ids, svc_data, transform, triplets_per_user=150):
        self.transform = transform
        self.triplets = []
        for uid in user_ids:
            g = svc_data[uid]['genuine']
            f = svc_data[uid]['forged']
            if len(g) < 2 or len(f) < 1:
                continue
            for _ in range(triplets_per_user):
                a, p = random.sample(g, 2)
                n = random.choice(f)
                self.triplets.append((a, p, n))
            # Also add cross-user negatives for diversity
            other_uids = [u for u in user_ids if u != uid]
            for _ in range(triplets_per_user // 3):
                a, p = random.sample(g, 2)
                other_uid = random.choice(other_uids)
                n = random.choice(svc_data[other_uid]['genuine'])
                self.triplets.append((a, p, n))
        random.shuffle(self.triplets)

    def __len__(self):
        return len(self.triplets)

    def __getitem__(self, idx):
        a_path, p_path, n_path = self.triplets[idx]
        a = self.transform(load_img_3ch(a_path))
        p = self.transform(load_img_3ch(p_path))
        n = self.transform(load_img_3ch(n_path))
        return a, p, n

# ── Tamper Dataset (clean synthetic only — NO SVC mixing) ──
class TamperDataset(Dataset):
    """Binary: clean (0) vs tampered (1).
    IMPORTANT: Only uses synthetic data to avoid domain confusion.
    """
    def __init__(self, clean_paths, tamper_paths, transform):
        self.transform = transform
        self.samples = [(p, 0) for p in clean_paths] + [(p, 1) for p in tamper_paths]
        random.shuffle(self.samples)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = load_img_3ch(path)
        return self.transform(img), torch.tensor(label, dtype=torch.long)

# ── Build DataLoaders ──
train_triplets = TripletDataset(train_uids, svc_users, train_tf, CFG['triplets_per_user'])
val_triplets = TripletDataset(val_uids, svc_users, val_tf, CFG['triplets_per_user'])

# Split tamper dataset 80/20
tamper_full = TamperDataset(clean_imgs, tampered_imgs, train_tf)
t_train_len = int(0.8 * len(tamper_full))
t_val_len = len(tamper_full) - t_train_len
tamper_train, tamper_val = random_split(tamper_full, [t_train_len, t_val_len])

siamese_train_loader = DataLoader(train_triplets, batch_size=CFG['batch_size'],
                                   shuffle=True, num_workers=CFG['workers'], pin_memory=True)
siamese_val_loader = DataLoader(val_triplets, batch_size=CFG['batch_size'],
                                 shuffle=False, num_workers=CFG['workers'], pin_memory=True)
tamper_train_loader = DataLoader(tamper_train, batch_size=CFG['batch_size'],
                                 shuffle=True, num_workers=CFG['workers'], pin_memory=True)
tamper_val_loader = DataLoader(tamper_val, batch_size=CFG['batch_size'],
                               shuffle=False, num_workers=CFG['workers'], pin_memory=True)

print(f'\n📊 DataLoader Summary:')
print(f'  Siamese train triplets: {len(train_triplets)}')
print(f'  Siamese val triplets: {len(val_triplets)}')
print(f'  Tamper train: {t_train_len}, val: {t_val_len}')
print('✅ Datasets ready')


In [ ]:
# ╔══════════════════════════════════════╗
# ║  CELL 4: Model Definitions           ║
# ╚══════════════════════════════════════╝

class SiameseEmbedder(nn.Module):
    """ResNet-18 pretrained backbone → 256-dim normalized embeddings.
    Uses 3-channel input to leverage ImageNet features.
    """
    def __init__(self, emb_dim=256):
        super().__init__()
        base = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        in_features = base.fc.in_features
        base.fc = nn.Identity()
        self.backbone = base
        self.head = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(512, emb_dim)
        )

    def forward(self, x):
        feat = self.backbone(x)
        emb = self.head(feat)
        return F.normalize(emb, p=2, dim=1)

class TamperDetector(nn.Module):
    """ResNet-18 pretrained → binary classifier (clean vs tampered).
    Leverages pretrained features for pixel-level anomaly detection.
    """
    def __init__(self):
        super().__init__()
        base = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        in_features = base.fc.in_features
        base.fc = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(in_features, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(128, 2)
        )
        self.model = base

    def forward(self, x):
        return self.model(x)

siamese_model = SiameseEmbedder(emb_dim=CFG['embedding_dim']).to(DEVICE)
tamper_model = TamperDetector().to(DEVICE)

s_params = sum(p.numel() for p in siamese_model.parameters())
t_params = sum(p.numel() for p in tamper_model.parameters())
print(f'✅ Models initialized')
print(f'  Siamese params: {s_params:,}')
print(f'  Tamper params: {t_params:,}')


In [ ]:
# ╔══════════════════════════════════════╗
# ║  CELL 5: Train Siamese (Triplet Loss)║
# ╚══════════════════════════════════════╝

triplet_loss_fn = nn.TripletMarginLoss(margin=CFG['triplet_margin'], p=2)

# Freeze backbone for first 3 epochs, then unfreeze
for param in siamese_model.backbone.parameters():
    param.requires_grad = False

opt_s = torch.optim.AdamW(filter(lambda p: p.requires_grad, siamese_model.parameters()),
                           lr=CFG['siamese_lr'], weight_decay=CFG['weight_decay'])

best_val_loss = float('inf')
patience_counter = 0
hist_s = {'train_loss': [], 'val_loss': []}

print(f'🏋️ Training Siamese Network ({CFG["siamese_epochs"]} epochs max)...')
print(f'   Triplet margin: {CFG["triplet_margin"]}')
print(f'   Freezing backbone for first 3 epochs\n')

for epoch in range(CFG['siamese_epochs']):
    # Unfreeze backbone after epoch 3
    if epoch == 3:
        print('   🔓 Unfreezing backbone...')
        for param in siamese_model.backbone.parameters():
            param.requires_grad = True
        opt_s = torch.optim.AdamW(siamese_model.parameters(),
                                   lr=CFG['siamese_lr'] * 0.1,  # lower LR for fine-tuning
                                   weight_decay=CFG['weight_decay'])
        sch_s = torch.optim.lr_scheduler.CosineAnnealingLR(opt_s, T_max=CFG['siamese_epochs'] - 3)

    siamese_model.train()
    train_losses = []
    for anchor, pos, neg in siamese_train_loader:
        anchor, pos, neg = anchor.to(DEVICE), pos.to(DEVICE), neg.to(DEVICE)
        opt_s.zero_grad()
        e_a = siamese_model(anchor)
        e_p = siamese_model(pos)
        e_n = siamese_model(neg)
        loss = triplet_loss_fn(e_a, e_p, e_n)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(siamese_model.parameters(), 1.0)
        opt_s.step()
        train_losses.append(loss.item())

    # Validation
    siamese_model.eval()
    val_losses = []
    with torch.no_grad():
        for anchor, pos, neg in siamese_val_loader:
            anchor, pos, neg = anchor.to(DEVICE), pos.to(DEVICE), neg.to(DEVICE)
            e_a = siamese_model(anchor)
            e_p = siamese_model(pos)
            e_n = siamese_model(neg)
            loss = triplet_loss_fn(e_a, e_p, e_n)
            val_losses.append(loss.item())

    tr_loss = np.mean(train_losses)
    va_loss = np.mean(val_losses)
    hist_s['train_loss'].append(tr_loss)
    hist_s['val_loss'].append(va_loss)

    if epoch >= 3:
        sch_s.step()

    lr = opt_s.param_groups[0]['lr']
    print(f'  Epoch {epoch+1:02d} | Train Loss: {tr_loss:.4f} | Val Loss: {va_loss:.4f} | LR: {lr:.6f}')

    if va_loss < best_val_loss:
        best_val_loss = va_loss
        patience_counter = 0
        torch.save(siamese_model.state_dict(), 'saved_models/siamese_best.pth')
    else:
        patience_counter += 1
        if patience_counter >= CFG['early_stop_patience']:
            print(f'  ⏹️ Early stopping at epoch {epoch+1}')
            break

siamese_model.load_state_dict(torch.load('saved_models/siamese_best.pth', map_location=DEVICE))
print(f'\n✅ Siamese training complete. Best val loss: {best_val_loss:.4f}')


In [ ]:
# ╔══════════════════════════════════════╗
# ║  CELL 6: Calibrate Siamese Threshold ║
# ╚══════════════════════════════════════╝

print('🔧 Calibrating Siamese threshold...')
siamese_model.eval()

genuine_sims = []
impostor_sims = []

# Compute similarity distributions on validation users
with torch.no_grad():
    for uid in val_uids:
        g = svc_users[uid]['genuine']
        f = svc_users[uid]['forged']

        # Embed all genuine
        g_embs = []
        for p in g:
            img = val_tf(load_img_3ch(p)).unsqueeze(0).to(DEVICE)
            g_embs.append(siamese_model(img))

        # Genuine-genuine pairs
        for i in range(len(g_embs)):
            for j in range(i+1, len(g_embs)):
                s = F.cosine_similarity(g_embs[i], g_embs[j]).item()
                genuine_sims.append(s)

        # Genuine-forged pairs
        for fp in f:
            f_img = val_tf(load_img_3ch(fp)).unsqueeze(0).to(DEVICE)
            f_emb = siamese_model(f_img)
            for ge in g_embs:
                s = F.cosine_similarity(ge, f_emb).item()
                impostor_sims.append(s)

genuine_sims = np.array(genuine_sims)
impostor_sims = np.array(impostor_sims)

print(f'  Genuine pairs: {len(genuine_sims)}, mean={genuine_sims.mean():.4f}, std={genuine_sims.std():.4f}')
print(f'  Impostor pairs: {len(impostor_sims)}, mean={impostor_sims.mean():.4f}, std={impostor_sims.std():.4f}')
print(f'  Gap: {genuine_sims.mean() - impostor_sims.mean():.4f}')

# Find optimal threshold via EER
all_scores = np.concatenate([genuine_sims, impostor_sims])
all_labels = np.concatenate([np.ones(len(genuine_sims)), np.zeros(len(impostor_sims))])

fpr, tpr, thresholds = roc_curve(all_labels, all_scores)
fnr = 1 - tpr
eer_idx = np.nanargmin(np.abs(fpr - fnr))
eer = (fpr[eer_idx] + fnr[eer_idx]) / 2
eer_threshold = thresholds[eer_idx]

# Also find threshold for best F1
best_f1, best_f1_th = 0, 0.5
for t in np.linspace(0.3, 0.99, 500):
    preds = (all_scores >= t).astype(int)
    f1 = f1_score(all_labels, preds, zero_division=0)
    if f1 > best_f1:
        best_f1 = f1
        best_f1_th = t

# Use the F1-optimal threshold (more practical than EER)
SIAMESE_THRESHOLD = float(best_f1_th)

print(f'\n📊 Siamese Calibration:')
print(f'  EER: {eer:.4f} ({eer*100:.1f}%)')
print(f'  EER threshold: {eer_threshold:.4f}')
print(f'  Best F1 threshold: {best_f1_th:.4f} (F1={best_f1:.4f})')
print(f'  ✅ Using threshold: {SIAMESE_THRESHOLD:.4f}')

# Verify with the chosen threshold
preds = (all_scores >= SIAMESE_THRESHOLD).astype(int)
acc = accuracy_score(all_labels, preds)
far = np.sum((preds == 1) & (all_labels == 0)) / max(np.sum(all_labels == 0), 1)
frr = np.sum((preds == 0) & (all_labels == 1)) / max(np.sum(all_labels == 1), 1)
print(f'  Accuracy: {acc:.4f}')
print(f'  FAR: {far:.4f}, FRR: {frr:.4f}')


In [ ]:
# ╔══════════════════════════════════════╗
# ║  CELL 7: Train Tamper Detector       ║
# ╚══════════════════════════════════════╝

# Freeze backbone for 2 epochs then fine-tune
for param in tamper_model.model.parameters():
    param.requires_grad = False
for param in tamper_model.model.fc.parameters():
    param.requires_grad = True

criterion_t = nn.CrossEntropyLoss()
opt_t = torch.optim.AdamW(filter(lambda p: p.requires_grad, tamper_model.parameters()),
                           lr=CFG['tamper_lr'], weight_decay=CFG['weight_decay'])

best_val_acc = 0.0
patience_counter = 0
hist_t = {'train_acc': [], 'val_acc': [], 'train_loss': [], 'val_loss': []}

print(f'🏋️ Training Tamper Detector ({CFG["tamper_epochs"]} epochs max)...')
print(f'   Freezing backbone for first 2 epochs\n')

for epoch in range(CFG['tamper_epochs']):
    if epoch == 2:
        print('   🔓 Unfreezing backbone...')
        for param in tamper_model.model.parameters():
            param.requires_grad = True
        opt_t = torch.optim.AdamW(tamper_model.parameters(),
                                   lr=CFG['tamper_lr'] * 0.1,
                                   weight_decay=CFG['weight_decay'])
        sch_t = torch.optim.lr_scheduler.CosineAnnealingLR(opt_t, T_max=CFG['tamper_epochs'] - 2)

    tamper_model.train()
    tr_loss, tr_correct, tr_total = 0.0, 0, 0
    for x, y in tamper_train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        opt_t.zero_grad()
        out = tamper_model(x)
        loss = criterion_t(out, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(tamper_model.parameters(), 1.0)
        opt_t.step()
        tr_loss += loss.item() * x.size(0)
        tr_correct += (out.argmax(1) == y).sum().item()
        tr_total += x.size(0)

    tamper_model.eval()
    va_loss, va_correct, va_total = 0.0, 0, 0
    with torch.no_grad():
        for x, y in tamper_val_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            out = tamper_model(x)
            loss = criterion_t(out, y)
            va_loss += loss.item() * x.size(0)
            va_correct += (out.argmax(1) == y).sum().item()
            va_total += x.size(0)

    tr_acc = tr_correct / max(tr_total, 1)
    va_acc = va_correct / max(va_total, 1)
    hist_t['train_acc'].append(tr_acc)
    hist_t['val_acc'].append(va_acc)
    hist_t['train_loss'].append(tr_loss / max(tr_total, 1))
    hist_t['val_loss'].append(va_loss / max(va_total, 1))

    if epoch >= 2:
        sch_t.step()

    lr = opt_t.param_groups[0]['lr']
    print(f'  Epoch {epoch+1:02d} | Train Acc: {tr_acc:.4f} | Val Acc: {va_acc:.4f} | Val Loss: {va_loss/max(va_total,1):.4f} | LR: {lr:.6f}')

    if va_acc > best_val_acc:
        best_val_acc = va_acc
        patience_counter = 0
        torch.save(tamper_model.state_dict(), 'saved_models/tamper_best.pth')
    else:
        patience_counter += 1
        if patience_counter >= CFG['early_stop_patience']:
            print(f'  ⏹️ Early stopping at epoch {epoch+1}')
            break

tamper_model.load_state_dict(torch.load('saved_models/tamper_best.pth', map_location=DEVICE))
print(f'\n✅ Tamper training complete. Best val accuracy: {best_val_acc:.4f} ({best_val_acc*100:.1f}%)')


In [ ]:
# ╔══════════════════════════════════════╗
# ║  CELL 8: Calibrate Tamper Threshold  ║
# ║         + Evaluation Plots           ║
# ╚══════════════════════════════════════╝

print('🔧 Calibrating Tamper threshold...')
tamper_model.eval()
all_probs, all_labels_t = [], []
with torch.no_grad():
    for x, y in tamper_val_loader:
        x = x.to(DEVICE)
        out = tamper_model(x)
        p = F.softmax(out, dim=1)[:, 1].cpu().numpy()
        all_probs.extend(p.tolist())
        all_labels_t.extend(y.numpy().tolist())

all_probs = np.array(all_probs)
all_labels_t = np.array(all_labels_t)

# Find threshold with FPR <= 3% and maximize recall
best_th, best_rec = 0.5, 0
for t in np.linspace(0.1, 0.99, 500):
    pred = (all_probs >= t).astype(int)
    fp = np.sum((pred == 1) & (all_labels_t == 0))
    tn = np.sum((pred == 0) & (all_labels_t == 0))
    tp = np.sum((pred == 1) & (all_labels_t == 1))
    fn = np.sum((pred == 0) & (all_labels_t == 1))
    fpr = fp / max(fp + tn, 1)
    rec = tp / max(tp + fn, 1)
    if fpr <= 0.03 and rec > best_rec:
        best_rec = rec
        best_th = float(t)

TAMPER_THRESHOLD = best_th

# Print report with calibrated threshold
pred_t = (all_probs >= TAMPER_THRESHOLD).astype(int)
print(f'\n📊 Tamper Detection Results @ threshold={TAMPER_THRESHOLD:.3f}:')
print(classification_report(all_labels_t, pred_t, target_names=['Clean', 'Tampered']))

tamper_acc = accuracy_score(all_labels_t, pred_t)
print(f'  Accuracy: {tamper_acc:.4f} ({tamper_acc*100:.1f}%)')
print(f'  Recall (tampered): {best_rec:.4f}')

# ── PLOTS ──
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Siamese Loss
axes[0,0].plot(hist_s['train_loss'], label='Train', linewidth=2)
axes[0,0].plot(hist_s['val_loss'], label='Val', linewidth=2)
axes[0,0].set_title('Siamese Triplet Loss', fontsize=13)
axes[0,0].set_xlabel('Epoch'); axes[0,0].set_ylabel('Loss')
axes[0,0].legend(); axes[0,0].grid(True, alpha=0.3)

# 2. Score Distribution
axes[0,1].hist(genuine_sims, bins=50, alpha=0.7, label='Genuine', color='green', density=True)
axes[0,1].hist(impostor_sims, bins=50, alpha=0.7, label='Impostor', color='red', density=True)
axes[0,1].axvline(SIAMESE_THRESHOLD, color='blue', linestyle='--', linewidth=2, label=f'Threshold={SIAMESE_THRESHOLD:.3f}')
axes[0,1].set_title('Siamese Score Distribution', fontsize=13)
axes[0,1].legend(); axes[0,1].grid(True, alpha=0.3)

# 3. Tamper Accuracy
axes[1,0].plot(hist_t['train_acc'], label='Train Acc', linewidth=2)
axes[1,0].plot(hist_t['val_acc'], label='Val Acc', linewidth=2)
axes[1,0].set_title('Tamper CNN Accuracy', fontsize=13)
axes[1,0].set_xlabel('Epoch'); axes[1,0].set_ylabel('Accuracy')
axes[1,0].legend(); axes[1,0].grid(True, alpha=0.3)

# 4. Confusion Matrix
cm = confusion_matrix(all_labels_t, pred_t)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Clean', 'Tampered'], yticklabels=['Clean', 'Tampered'],
            ax=axes[1,1], annot_kws={'size': 14})
axes[1,1].set_title(f'Tamper Confusion Matrix @ th={TAMPER_THRESHOLD:.3f}', fontsize=13)

plt.tight_layout()
plt.savefig('results/v8_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

# Save calibration
calib = {
    'siamese_threshold': SIAMESE_THRESHOLD,
    'tamper_threshold': TAMPER_THRESHOLD,
    'siamese_eer': float(eer),
    'tamper_val_acc': float(best_val_acc),
    'cfg': CFG
}
torch.save(calib, 'saved_models/calibration.pth')

print(f'\n══════════════════════════════════════')
print(f'  📋 FINAL CALIBRATION SUMMARY')
print(f'══════════════════════════════════════')
print(f'  Siamese threshold: {SIAMESE_THRESHOLD:.4f}')
print(f'  Siamese EER: {eer:.4f} ({eer*100:.1f}%)')
print(f'  Genuine mean sim: {genuine_sims.mean():.4f}')
print(f'  Impostor mean sim: {impostor_sims.mean():.4f}')
print(f'  Tamper threshold: {TAMPER_THRESHOLD:.4f}')
print(f'  Tamper val accuracy: {best_val_acc:.4f}')
print(f'══════════════════════════════════════')


In [ ]:
# ╔══════════════════════════════════════╗
# ║  CELL 9: Database + Verification     ║
# ║          Engine + Flask API           ║
# ╚══════════════════════════════════════╝

from flask import Flask, request, jsonify
from flask_cors import CORS
import jwt as pyjwt

DB_PATH = 'sigauth.db'
JWT_SECRET = 'sigauth_jwt_secret_change_in_prod'

# ── Database ──
def init_db():
    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    c.execute('''CREATE TABLE IF NOT EXISTS users (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        username TEXT UNIQUE NOT NULL,
        email TEXT UNIQUE NOT NULL,
        password_hash TEXT NOT NULL,
        role TEXT DEFAULT 'user',
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )''')
    c.execute('''CREATE TABLE IF NOT EXISTS reference_signatures (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        person_name TEXT NOT NULL,
        image_data BLOB NOT NULL,
        image_hash TEXT,
        filename TEXT,
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )''')
    c.execute('''CREATE TABLE IF NOT EXISTS verification_logs (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        user_id INTEGER,
        person_name TEXT,
        result TEXT,
        confidence REAL,
        similarity_score REAL,
        tamper_score REAL,
        details TEXT,
        timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )''')
    # Default admin
    admin_hash = hashlib.sha256('admin123'.encode()).hexdigest()
    try:
        c.execute('INSERT INTO users (username, email, password_hash, role) VALUES (?, ?, ?, ?)',
                  ('admin', 'admin@sigauth.com', admin_hash, 'admin'))
    except sqlite3.IntegrityError:
        pass
    conn.commit()
    conn.close()

init_db()
print('✅ Database initialized')

# ── Inference transform (must match training val_tf) ──
inference_tf = val_tf

# ── Verification Engine ──
class VerificationEngine:
    def __init__(self, siamese, tamper, s_threshold, t_threshold, device):
        self.siamese = siamese.eval()
        self.tamper = tamper.eval()
        self.s_th = float(s_threshold)
        self.t_th = float(t_threshold)
        self.device = device

    def _decode_b64(self, b64_str: str) -> Tuple[bytes, Image.Image]:
        if ',' in b64_str:
            b64_str = b64_str.split(',', 1)[1]
        raw = base64.b64decode(b64_str)
        img = Image.open(io.BytesIO(raw)).convert('L')
        return raw, img

    def _to_tensor(self, gray_img: Image.Image) -> torch.Tensor:
        rgb = Image.merge('RGB', (gray_img, gray_img, gray_img))
        return inference_tf(rgb).unsqueeze(0).to(self.device)

    def _compute_stroke_features(self, img: Image.Image) -> dict:
        """Extract handcrafted stroke-level features for detailed analysis."""
        arr = np.array(img.resize((224, 224)), dtype=np.float32) / 255.0
        # Binarize
        binary = (arr < 0.5).astype(np.float32)
        ink_ratio = binary.mean()

        # Horizontal/vertical projection profiles
        h_proj = binary.sum(axis=1)
        v_proj = binary.sum(axis=0)
        h_smoothness = 1.0 - np.std(np.diff(h_proj)) / max(np.mean(h_proj), 1e-6)
        v_smoothness = 1.0 - np.std(np.diff(v_proj)) / max(np.mean(v_proj), 1e-6)

        # Edge density (Sobel-like)
        dx = np.abs(np.diff(arr, axis=1))
        dy = np.abs(np.diff(arr, axis=0))
        edge_density = (dx.mean() + dy.mean()) / 2

        # Local variance (texture consistency)
        from PIL import ImageFilter as IF
        blurred = np.array(img.resize((224,224)).filter(IF.GaussianBlur(3)), dtype=np.float32) / 255.0
        local_var = np.mean((arr - blurred) ** 2)

        return {
            'ink_ratio': float(ink_ratio),
            'h_smoothness': float(np.clip(h_smoothness, 0, 1)),
            'v_smoothness': float(np.clip(v_smoothness, 0, 1)),
            'edge_density': float(edge_density),
            'local_variance': float(local_var)
        }

    def verify(self, test_b64: str, ref_image_bytes_list: List[bytes], ref_hashes: List[str]) -> dict:
        start = time.time()
        raw_bytes, test_gray = self._decode_b64(test_b64)
        test_hash = hashlib.md5(raw_bytes).hexdigest()

        # ── Exact match shortcut ──
        if test_hash in set(ref_hashes):
            elapsed = int((time.time() - start) * 1000)
            return {
                'result': 'authentic',
                'confidence': 1.0,
                'similarity_score': 1.0,
                'tamper_score': 0.0,
                'processing_time_ms': elapsed,
                'details': {
                    'method': 'exact_hash_match',
                    'stroke_consistency': 1.0,
                    'pressure_pattern': 1.0,
                    'spatial_alignment': 1.0,
                    'pixel_anomalies': 0.0,
                    'tamper_probability': 0.0
                }
            }

        test_t = self._to_tensor(test_gray)
        stroke_feat = self._compute_stroke_features(test_gray)

        with torch.no_grad():
            # ── Step 1: Tamper detection ──
            tamper_logits = self.tamper(test_t)
            tamper_probs = F.softmax(tamper_logits, dim=1)
            tamper_prob = tamper_probs[0, 1].item()  # P(tampered)

            # ── Step 2: Siamese similarity ──
            test_emb = self.siamese(test_t)
            best_sim = -1.0
            for ref_bytes in ref_image_bytes_list:
                ref_gray = Image.open(io.BytesIO(ref_bytes)).convert('L')
                ref_t = self._to_tensor(ref_gray)
                ref_emb = self.siamese(ref_t)
                sim = F.cosine_similarity(test_emb, ref_emb).item()
                best_sim = max(best_sim, sim)

        # ── Step 3: Decision Logic (simple hard rules) ──
        # Rule 1: If tamper score is very high → TAMPERED
        # Rule 2: If not tampered and similarity >= threshold → AUTHENTIC
        # Rule 3: If not tampered and similarity < threshold → FORGED

        is_tampered = tamper_prob >= self.t_th
        is_similar = best_sim >= self.s_th

        if is_tampered:
            result = 'tampered'
            confidence = tamper_prob
        elif is_similar:
            result = 'authentic'
            # Confidence scales with how far above threshold
            confidence = 0.7 + 0.3 * min(1.0, (best_sim - self.s_th) / (1.0 - self.s_th + 1e-6))
        else:
            result = 'forged'
            confidence = 0.7 + 0.3 * min(1.0, (self.s_th - best_sim) / (self.s_th + 1e-6))

        # ── Compute display features ──
        # Stroke consistency: high for authentic, low for forged
        stroke_consistency = float(np.clip(best_sim * 0.85 + stroke_feat['h_smoothness'] * 0.15, 0, 1))
        # Pressure pattern: derived from edge density and similarity
        pressure_pattern = float(np.clip(best_sim * 0.7 + (1 - stroke_feat['edge_density'] * 5) * 0.3, 0, 1))
        # Spatial alignment: cosine sim itself is a good proxy
        spatial_alignment = float(np.clip(best_sim, 0, 1))
        # Pixel anomalies: tamper probability
        pixel_anomalies = float(tamper_prob)

        elapsed = int((time.time() - start) * 1000)

        return {
            'result': result,
            'confidence': round(float(confidence), 4),
            'similarity_score': round(float(best_sim), 4),
            'tamper_score': round(float(tamper_prob), 4),
            'processing_time_ms': elapsed,
            'details': {
                'stroke_consistency': round(stroke_consistency, 4),
                'pressure_pattern': round(pressure_pattern, 4),
                'spatial_alignment': round(spatial_alignment, 4),
                'pixel_anomalies': round(pixel_anomalies, 4),
                'tamper_probability': round(float(tamper_prob), 4),
                'thresholds': {
                    'siamese': round(self.s_th, 4),
                    'tamper': round(self.t_th, 4)
                },
                'raw_features': stroke_feat
            }
        }

engine = VerificationEngine(siamese_model, tamper_model, SIAMESE_THRESHOLD, TAMPER_THRESHOLD, DEVICE)
print('✅ Verification engine ready')

# ── Auth helpers ──
def hash_pw(pw: str) -> str:
    return hashlib.sha256(pw.encode()).hexdigest()

def make_token(user_id: int, role: str) -> str:
    payload = {'user_id': user_id, 'role': role, 'exp': int(time.time()) + 86400}
    return pyjwt.encode(payload, JWT_SECRET, algorithm='HS256')

def decode_token(tok: str):
    try:
        p = pyjwt.decode(tok, JWT_SECRET, algorithms=['HS256'])
        return p.get('user_id'), p.get('role')
    except Exception:
        return None, None

def get_current_user():
    auth = request.headers.get('Authorization', '')
    if not auth.startswith('Bearer '):
        return None
    uid, role = decode_token(auth[7:])
    if uid is None:
        return None
    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    c.execute('SELECT id, username, email, role FROM users WHERE id=?', (uid,))
    row = c.fetchone()
    conn.close()
    if not row:
        return None
    return {'id': row[0], 'username': row[1], 'email': row[2], 'role': row[3]}

# ══════════════════════════════════
# Flask API
# ══════════════════════════════════

app = Flask(__name__)
CORS(app, resources={r'/*': {'origins': '*'}})

# ── Health ──
@app.route('/health', methods=['GET'])
def health():
    return jsonify({
        'status': 'healthy',
        'models': {'siamese': 'loaded', 'tamper': 'loaded'},
        'thresholds': {'siamese': SIAMESE_THRESHOLD, 'tamper': TAMPER_THRESHOLD}
    })

# ── Auth: Signup ──
@app.route('/auth/signup', methods=['POST'])
def signup():
    d = request.get_json(force=True)
    username = d.get('username', '').strip()
    email = d.get('email', '').strip().lower()
    password = d.get('password', '')
    if not username or not email or len(password) < 6:
        return jsonify({'error': 'Username, email, and password (6+ chars) required'}), 400
    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    try:
        c.execute('INSERT INTO users (username, email, password_hash) VALUES (?, ?, ?)',
                  (username, email, hash_pw(password)))
        conn.commit()
        uid = c.lastrowid
    except sqlite3.IntegrityError:
        conn.close()
        return jsonify({'error': 'Username or email already exists'}), 409
    conn.close()
    token = make_token(uid, 'user')
    return jsonify({'token': token, 'user': {'id': uid, 'username': username, 'email': email, 'role': 'user'}})

# ── Auth: Login ──
@app.route('/auth/login', methods=['POST'])
def login():
    d = request.get_json(force=True)
    email = d.get('email', '').strip().lower()
    password = d.get('password', '')
    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    c.execute('SELECT id, username, email, password_hash, role FROM users WHERE email=?', (email,))
    row = c.fetchone()
    conn.close()
    if not row or row[3] != hash_pw(password):
        return jsonify({'error': 'Invalid credentials'}), 401
    token = make_token(row[0], row[4])
    return jsonify({'token': token, 'user': {'id': row[0], 'username': row[1], 'email': row[2], 'role': row[4]}})

# ── Auth: Profile ──
@app.route('/auth/profile', methods=['GET'])
def profile():
    user = get_current_user()
    if not user:
        return jsonify({'error': 'Unauthorized'}), 401
    return jsonify({'user': user})

# ── Persons list ──
@app.route('/persons', methods=['GET'])
def list_persons():
    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    c.execute('SELECT MIN(id), person_name, COUNT(*) FROM reference_signatures GROUP BY person_name ORDER BY person_name')
    rows = c.fetchall()
    conn.close()
    return jsonify({'persons': [{'id': r[0], 'person_name': r[1], 'image_count': r[2]} for r in rows]})

# ── Verify ──
@app.route('/verify', methods=['POST'])
def verify_route():
    d = request.get_json(force=True)
    person_name = d.get('person_name', '').strip()
    test_image = d.get('test_image', '')
    if not person_name or not test_image:
        return jsonify({'error': 'person_name and test_image required'}), 400

    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    c.execute("SELECT image_data, COALESCE(image_hash, '') FROM reference_signatures WHERE person_name=?", (person_name,))
    rows = c.fetchall()
    if not rows:
        conn.close()
        return jsonify({'error': f'No reference signatures found for \"{person_name}\"'}), 404

    ref_bytes = [r[0] for r in rows]
    ref_hashes = [r[1] for r in rows if r[1]]

    result = engine.verify(test_image, ref_bytes, ref_hashes)

    # Log
    user = get_current_user()
    c.execute('''INSERT INTO verification_logs
                 (user_id, person_name, result, confidence, similarity_score, tamper_score, details)
                 VALUES (?, ?, ?, ?, ?, ?, ?)''',
              (user['id'] if user else None, person_name,
               result['result'], result['confidence'],
               result['similarity_score'], result['tamper_score'],
               json.dumps(result['details'])))
    conn.commit()
    conn.close()
    return jsonify(result)

# ── Admin: Add signatures ──
@app.route('/admin/signatures/add', methods=['POST'])
def admin_add_signature():
    user = get_current_user()
    if not user or user['role'] != 'admin':
        return jsonify({'error': 'Admin access required'}), 403

    person_name = request.form.get('person_name', '').strip()
    files = request.files.getlist('signatures')
    if not person_name or not files:
        return jsonify({'error': 'person_name and signature files required'}), 400

    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    count = 0
    for f in files:
        data = f.read()
        if not data:
            continue
        h = hashlib.md5(data).hexdigest()
        c.execute('INSERT INTO reference_signatures (person_name, image_data, image_hash, filename) VALUES (?, ?, ?, ?)',
                  (person_name, data, h, f.filename))
        count += 1
    conn.commit()
    conn.close()
    return jsonify({'message': f'Added {count} signature(s) for {person_name}', 'count': count})

# ── Admin: List signatures ──
@app.route('/admin/signatures', methods=['GET'])
def admin_list_signatures():
    user = get_current_user()
    if not user or user['role'] != 'admin':
        return jsonify({'error': 'Admin access required'}), 403
    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    c.execute('''SELECT MIN(id) as id, person_name, COUNT(*) as image_count, MIN(created_at) as created_at
                 FROM reference_signatures GROUP BY person_name ORDER BY person_name''')
    rows = c.fetchall()
    conn.close()
    return jsonify({'signatures': [{'id': r[0], 'person_name': r[1], 'image_count': r[2], 'created_at': r[3]} for r in rows]})

# ── Admin: Delete signatures for a person ──
@app.route('/admin/signatures/<int:sig_id>', methods=['DELETE'])
def admin_delete_signature(sig_id):
    user = get_current_user()
    if not user or user['role'] != 'admin':
        return jsonify({'error': 'Admin access required'}), 403
    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    # Get person_name from the id, then delete all for that person
    c.execute('SELECT person_name FROM reference_signatures WHERE id=?', (sig_id,))
    row = c.fetchone()
    if not row:
        conn.close()
        return jsonify({'error': 'Not found'}), 404
    c.execute('DELETE FROM reference_signatures WHERE person_name=?', (row[0],))
    conn.commit()
    conn.close()
    return jsonify({'message': f'Deleted all signatures for {row[0]}'})

# ── Admin: Verification logs ──
@app.route('/admin/logs', methods=['GET'])
def admin_logs():
    user = get_current_user()
    if not user or user['role'] != 'admin':
        return jsonify({'error': 'Admin access required'}), 403
    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    c.execute('''SELECT v.id, v.user_id, COALESCE(u.username, 'anonymous') as username,
                        v.person_name, v.result, v.confidence, v.similarity_score,
                        v.tamper_score, v.timestamp
                 FROM verification_logs v
                 LEFT JOIN users u ON v.user_id = u.id
                 ORDER BY v.timestamp DESC
                 LIMIT 200''')
    rows = c.fetchall()
    conn.close()
    logs = [{
        'id': r[0], 'user_id': r[1], 'username': r[2],
        'person_name': r[3], 'result': r[4], 'confidence': r[5],
        'similarity_score': r[6], 'tamper_score': r[7], 'timestamp': r[8]
    } for r in rows]
    return jsonify({'logs': logs})

print('✅ Flask API configured with all endpoints:')
print('   GET  /health')
print('   POST /auth/signup')
print('   POST /auth/login')
print('   GET  /auth/profile')
print('   GET  /persons')
print('   POST /verify')
print('   POST /admin/signatures/add')
print('   GET  /admin/signatures')
print('   DEL  /admin/signatures/<id>')
print('   GET  /admin/logs')


In [ ]:
# ╔══════════════════════════════════════╗
# ║  CELL 10: Start Server (ngrok)       ║
# ╚══════════════════════════════════════╝

# ⚠️ IMPORTANT: Paste your ngrok auth token below!
# Get one free at https://dashboard.ngrok.com/get-started/your-authtoken

NGROK_AUTH_TOKEN = ''  # ← Paste your token here
PORT = 5000

from pyngrok import ngrok

# Kill any existing tunnels
ngrok.kill()

if NGROK_AUTH_TOKEN:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)

public_url = ngrok.connect(PORT).public_url

print('\n' + '='*60)
print(f'🚀 SigAuth Backend is LIVE!')
print(f'='*60)
print(f'\n📡 Public URL: {public_url}')
print(f'\n👉 Copy this URL to your .env.local file:')
print(f'   VITE_API_URL={public_url}')
print(f'\n🔑 Admin login: admin@sigauth.com / admin123')
print(f'\n⚠️  Keep this Colab tab open for the server to stay running!')
print(f'='*60)

app.run(host='0.0.0.0', port=PORT, debug=False, use_reloader=False)


---

## 🎛️ Fine-Tuning Guide

### To increase accuracy, edit `CFG` in Cell 1:

| Parameter | Current | Try | Effect |
|-----------|---------|-----|--------|
| `siamese_epochs` | 40 | 60 | More training time for embedding quality |
| `tamper_epochs` | 50 | 70 | More training time for tamper detection |
| `siamese_lr` | 3e-4 | 1e-4 | Slower but more precise convergence |
| `tamper_lr` | 1e-4 | 5e-5 | Slower but more stable |
| `triplet_margin` | 0.5 | 0.7 | Forces larger gap between genuine/forged |
| `triplets_per_user` | 150 | 250 | More training data (but slower) |
| `batch_size` | 32 | 16 | More stable gradients (slower) |
| `early_stop_patience` | 10 | 15 | Allows longer training plateaus |
| `embedding_dim` | 256 | 512 | Richer embeddings (needs more data) |

### After changing, re-run:
1. Cell 1 (config)
2. Cell 3 (rebuild datasets)
3. Cell 4 (rebuild models)
4. Cells 5-8 (retrain + recalibrate)
5. Cell 9-10 (restart API)

### Key improvements in v8 over v7:
- **Pretrained ResNet-18**: Leverages ImageNet features instead of training from scratch
- **Triplet Loss**: Creates much larger genuine/impostor gap than Contrastive Loss
- **No domain mixing**: Tamper model only sees synthetic data, so real signatures don't get flagged as tampered
- **Hard-rule fusion**: No arbitrary softmax coefficients — just clear threshold-based decisions
- **All API endpoints**: Includes `/auth/profile`, `/admin/logs`, `DELETE /admin/signatures/<id>`
